# reshape-back — ex2: reshape_back across squeeze/unsqueeze — view-op family unification

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reshape-back`. Running the final beacon cell reports progress against the `Backprop: reshape_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: reshape_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reshape-back`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reshape-back"
DD_SUBTOPIC = "Backprop: reshape_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `reshape_back` across the squeeze/unsqueeze family — quick refresher

ex1 covered the canonical reshape `(2, 6) → (3, 4)`. The deeper facet: the size-1 axis manipulations — `squeeze`, `unsqueeze`, `reshape(N, 1) → (N,)` — are all special cases of `reshape_back`. None needs a separate back fn.

```
x.shape  = (N, 1)                          # column vector
out_a    = x.reshape(N)                    # squeeze to 1-D
out_b    = out_a.reshape(N, 1)             # unsqueeze back

grad_back_a = reshape_back(g, out_a, x,     (N,))    # → (N, 1)
grad_back_b = reshape_back(g, out_b, out_a, (N, 1))  # → (N,)
```

Both calls are `grad_out.reshape(x.shape)`. The same backward handles every view-op shape change. PyTorch's `view`, `flatten`, `squeeze`, `unsqueeze` all reduce to this primitive on the backward.

### Exercise 2 — reshape_back across squeeze/unsqueeze — view-op family unification

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply reshape_back uniformly to the squeeze/unsqueeze family: the same back fn handles (N, 1) ↔ (N,) ↔ (N, 1) round-trips.
> Keywords: reshape, squeeze, unsqueeze, view-op, shape-restore
> ```

**KCs targeted:** `reshape-backward-pattern`, `backward-fn-signature`

Implement `reshape_back(grad_out, out, x, new_shape)` AND a chain `squeeze_unsqueeze_chain(grad_out, x_leaf)` that walks the reverse pass of:

```
x_leaf.shape  = (N, 1)
u    = x_leaf.reshape(N)            # squeeze
y    = u.reshape(N, 1)              # unsqueeze
```

The chain calls `reshape_back` TWICE, once for each step. Both calls reduce to `grad_out.reshape(x.shape)` — the same primitive handles both squeeze and unsqueeze, because they're both pure view operations.

Tests verify:
- `reshape_back` works for `(N,) → (N, 1)` and `(N, 1) → (N,)` (both `unsqueeze`-like and `squeeze`-like reshapes),
- the two-step chain returns `grad_out` reshaped to `(N, 1)` (matching `x_leaf.shape`),
- agreement with torch.autograd on the equivalent chain.

No autograd. The point is that `reshape_back` SUBSUMES `squeeze_back` and `unsqueeze_back` — no separate back fn needed.

In [ ]:
def reshape_back(grad_out: Tensor, out: Tensor, x: Tensor, new_shape: tuple) -> Tensor:
    return grad_out.reshape(x.shape)


def squeeze_unsqueeze_chain(grad_out: Tensor, x_leaf: Tensor) -> Tensor:
    # forward (cached for back-fn use)
    N = x_leaf.shape[0]
    u = x_leaf.reshape(N)             # (N,)
    y = u.reshape(N, 1)               # (N, 1)
    # reverse
    g_u = reshape_back(grad_out, y, u, (N, 1))
    g_x = reshape_back(g_u, u, x_leaf, (N,))
    return g_x


<details><summary>Solution</summary>

```python
def reshape_back(grad_out: Tensor, out: Tensor, x: Tensor, new_shape: tuple) -> Tensor:
    return grad_out.reshape(x.shape)


def squeeze_unsqueeze_chain(grad_out: Tensor, x_leaf: Tensor) -> Tensor:
    # forward (cached for back-fn use)
    N = x_leaf.shape[0]
    u = x_leaf.reshape(N)             # (N,)
    y = u.reshape(N, 1)               # (N, 1)
    # reverse
    g_u = reshape_back(grad_out, y, u, (N, 1))
    g_x = reshape_back(g_u, u, x_leaf, (N,))
    return g_x
```

**Why the same back fn handles squeeze AND unsqueeze.** Both are pure view operations — same data, different shape interpretation. `grad_out.reshape(x.shape)` is the universal answer; the direction doesn't matter.

**Why bit-exact through the chain.** Two reshapes are pure storage re-interpretations — no arithmetic, no rounding. `t.equal` (not `allclose`) pins this down. Compare with `exp ∘ log` where rounding does creep in.

**Production note.** PyTorch's autograd has separate `SqueezeBackward` / `UnsqueezeBackward` / `ViewBackward` nodes for performance reasons (no shape-checking overhead), but the math is identical. In our MiniTensor where we want minimal surface area, one `reshape_back` covers the whole family.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()